# Practical 2 — Select architecture and spectral preprocessing
### Student exercises
**Sayan CHAKI · LIRIS · École Centrale de Lyon · Université Lyon 2**

**Objectives:** choose models with cross-validation, study spectral derivatives, and compare an MLP with partial least squares (PLS). **Time:** 75–90 minutes. **Total:** 20 marks.

## How to use this notebook
Upload this `.ipynb` through **Google Colab → File → Upload notebook**. Use a **CPU** runtime; no GPU or additional dataset download is required. Run cells from top to bottom. Standard Colab installations contain the required NumPy, pandas, SciPy, Matplotlib and scikit-learn packages.

**Student notebook:** replace each `NotImplementedError` with your code; later cells may depend on your answer. Write explanations in the answer cells. **Solution notebook:** complete code and suggested interpretations are provided. Results may vary slightly with library versions and optimizer convergence. Keep convergence warnings visible; if one appears, inspect stability before claiming convergence.

Prerequisites: NumPy arrays, basic plotting, functions, train/test splits, and a first introduction to MLPs. All notebooks are independently runnable. Suggested pace: three sessions of 60–90 minutes, plus optional extensions.


## Physical setting and measured data
You are calibrating an optical instrument to estimate composition without destroying a sample. A Tecator Infratec Food and Feed Analyzer measured **215 real near-infrared spectra** over **850–1050 nm**. Each sample has 100 absorbance channels and independently measured fat, water and protein percentages. These are measured spectra, not analytical curves synthesized for this practical.

The optical quantity is absorbance:
\[
A(\lambda)=-\log_{10} T(\lambda),\qquad T=I/I_0.
\]
Absorbance is dimensionless. For an ideal homogeneous absorbing medium, Beer–Lambert motivates a relationship between concentration and absorbance. Meat is a heterogeneous, scattering mixture: these exercises learn an empirical calibration, not an exact inversion of Beer–Lambert. Near-infrared bands overlap, and neighboring channels are highly correlated.

**Inputs:** the 100 spectral channels only. **Target:** fat content (%). Do not put water, protein, row numbers, or target values into the inputs. An error of 2 in this target is **2 percentage points (pp)**, not necessarily 2% relative error.

The data pool contains the original training, monitoring and test samples. We use a reproducible teaching split, not the original benchmark protocol. No acquisition-batch metadata is supplied: a random held-out split does not demonstrate generalization to another instrument, laboratory or batch.


## MLP reminder
For a single hidden layer:
\[
h=\tanh(W_1x+b_1),\qquad \hat y=W_2h+b_2.
\]
Here the input has 100 components and the output is one continuous value. Training adjusts weights and biases to reduce squared prediction error with L2 regularization. A linear output is appropriate for regression; a sigmoid would unnecessarily restrict the scaled target. We use `MLPRegressor`, with train-only feature and target scaling. L-BFGS is practical for this small dataset; it uses gradients from backpropagation but does not provide an epoch-by-epoch learning curve.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.signal import savgol_filter
from IPython.display import display
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)  # Small matrices: avoids excessive thread overhead.
SEED = 42
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.grid': True, 'font.size': 11})
print('scikit-learn:', sklearn.__version__)


## Load the real dataset
The next provided cell contains the compressed measurements. You do not need to edit it.

In [ ]:
# Real measured data, embedded for offline/reliable classroom use.
# This is an exact CSV conversion of modeldata's meats.RData, not generated data.
import base64, gzip, io, hashlib
_PAYLOAD = "..."
csv_bytes = gzip.decompress(base64.b64decode(_PAYLOAD))
assert hashlib.sha256(csv_bytes).hexdigest() == 'c8d0fa4920c56bb115cc6e53b9df5aa096bdf87edc3e5972dc98321dd7a21118'
df = pd.read_csv(io.BytesIO(csv_bytes))
spectral_columns = [f'x_{i:03d}' for i in range(1, 101)]
X = df[spectral_columns].to_numpy(dtype=float)
y = df['fat'].to_numpy(dtype=float)
wavelengths = np.linspace(850, 1050, 100)  # Equally spaced nominal grid, nm.
assert X.shape == (215, 100) and np.isfinite(X).all() and np.isfinite(y).all()
display(df.iloc[:5, -3:])
print('Spectra:', X.shape, '| Fat range (%):', (y.min(), y.max()))


In [ ]:
# Same sample-level split in all three notebooks: 172 development / 43 test.
# This teaching split replaces the source's original split; it is not a benchmark replication.
train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.2, random_state=SEED)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
assert set(train_idx).isdisjoint(test_idx)
print('Development:', len(train_idx), 'Test:', len(test_idx))


In [ ]:
def metrics(y_true, y_pred):
    y_pred = np.asarray(y_pred).ravel()
    return {'MAE (pp)': mean_absolute_error(y_true, y_pred),
            'RMSE (pp)': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred)}

def make_mlp(hidden=(32,), alpha=0.1, pre='passthrough', seed=SEED):
    # Scaling is learned only during fit. Target scaling is also train-only.
    pipeline = Pipeline([('pre', pre), ('scale', StandardScaler()),
                         ('mlp', MLPRegressor(hidden_layer_sizes=hidden,
                           activation='tanh', solver='lbfgs', alpha=alpha,
                           max_iter=5000, max_fun=100000, tol=1e-3, random_state=seed))])
    return TransformedTargetRegressor(regressor=pipeline, transformer=StandardScaler())


## 1. Spectral derivatives (4 marks)
Implement a first derivative with a Savitzky–Golay filter (window=11 channels, polynomial order=2, derivative=1). Use channel spacing in nm. Plot one spectrum and its derivative on separate axes. Explain what a derivative removes and why noise may be amplified. This transformation is applied separately to each sample.

In [ ]:
def spectral_derivative(A):
    # TODO: savgol_filter(..., axis=1, delta=...)
    raise NotImplementedError("Implement derivative")

**Your interpretation:**

_Write your answer here, referring to your plots and numerical results._

## 2. Compare architectures fairly (6 marks)
Run shuffled 3-fold CV on the development set only. Compare hidden layers (16,), (32,), (32,16), alpha values 0.01 and 1.0, and raw versus derivative spectra (12 configurations). Rank by CV RMSE. Report mean±fold standard deviation for the five best candidates. All learned preprocessing must remain inside the estimator. Explain why fold variation is not a confidence interval.

In [ ]:
# TODO: define cv, parameter grid, GridSearchCV, fit, and display ranked results.
raise NotImplementedError("Run leakage-free model selection")

**Your interpretation:**

_Write your answer here, referring to your plots and numerical results._

## 3. Compare with a spectroscopy baseline (4 marks)
PLS forms latent directions useful for predicting the response and is well suited to correlated spectra. Tune its component count over 2, 5, 10 and 15, using exactly the same folds and scoring as the MLP. Do not claim that neural networks must win.

In [ ]:
# TODO: fit a GridSearchCV for PLSRegression(scale=True).
raise NotImplementedError("Fit PLS comparator")

**Your interpretation:**

_Write your answer here, referring to your plots and numerical results._

## 4. Final evaluation and parameter budget (4 marks)
Freeze both searches. Evaluate their refitted best estimators once on the held-out test set. Count the selected MLP parameters from its fitted weight arrays. Explain why a model selected here must not be selected again by comparing test errors across all three practicals.

In [ ]:
# TODO: test metrics, parameter count, prediction plot.
raise NotImplementedError("Evaluate frozen searches")

**Your interpretation:**

_Write your answer here, referring to your plots and numerical results._

## Submission
Provide the CV table, selected settings, final test results and a 150-word comparison. **Optional extension:** repeat development-set CV with several initialization seeds while keeping the test set closed. Explain the distinction between initialization variability and split variability.

## Sources and data provenance
- Dataset documentation: https://modeldata.tidymodels.org/reference/meats.html
- Measured data source: https://raw.githubusercontent.com/tidymodels/modeldata/main/data/meats.RData
- Original description and redistribution note: https://raw.githubusercontent.com/tidymodels/modeldata/main/R/meats.R
- Wavelength-grid documentation: https://search.r-project.org/CRAN/refmans/fsemipar/help/Tecator.html
- MLP API: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html
- Pipelines and leakage: https://scikit-learn.org/stable/common_pitfalls.html

Retrieved 30 September 2026. Converted from the source R object to CSV without changing values or row order. CSV and original source description are also in the downloadable bundle. No synthetic samples are included in the source CSV. Perturbations in Practical 3 are explicitly simulated.

### Original dataset description and permission note (retained with redistribution)
Fat, water and protein content of meat samples
"These data are recorded on a Tecator Infratec Food and Feed Analyzer
working in the wavelength range 850 - 1050 nm by the Near Infrared
Transmission (NIT) principle. Each sample contains finely chopped pure meat
with different moisture, fat and protein contents.
If results from these data are used in a publication we want you to mention
the instrument and company name (Tecator) in the publication.  In addition,
please send a preprint of your article to:
Karin Thente, Tecator AB, Box 70, S-263 21 Hoganas, Sweden
The data are available in the public domain with no responsibility from the
original data source. The data can be redistributed as long as this
permission note is attached."
"For each meat sample the data consists of a 100 channel spectrum of
absorbances and the contents of moisture (water), fat and protein.  The
absorbance is -log10 of the transmittance measured by the spectrometer. The
three contents, measured in percent, are determined by analytic chemistry."
Included here are the training, monitoring and test sets.